In [56]:
from os import NGROUPS_MAX
import numpy as np
import random
from google.colab import files
import pandas as pd
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from google.colab import files

In [57]:
def get_shape(lst):
    if not isinstance(lst, list):
        # Não é uma lista, retorne uma tupla vazia
        return ()
    if not lst:
        # Lista vazia, retorne (0,)
        return (0,)
    shape = []
    sub_lst = lst
    while isinstance(sub_lst, list):
        shape.append(len(sub_lst))
        sub_lst = sub_lst[0] if sub_lst else []
    return tuple(shape)

def trocar_capacitancias(matrizes):
    idx1, idx2 = random.sample(range(len(matrizes)), 2)
    pos1 = (random.randrange(n_lin), random.randrange(n_col))
    pos2 = (random.randrange(n_lin), random.randrange(n_col))
    matrizes[idx1][pos1], matrizes[idx2][pos2] = matrizes[idx2][pos2], matrizes[idx1][pos1]

def diferenca_total_capacitancias(capacitancias):
    return np.ptp(capacitancias)

In [58]:
def capacitancia_equivalente_por_ramo(latas):
    inverso_soma_inversos = np.zeros(latas.shape[0])
    paralelo = np.zeros(latas.shape[1])
    for i in range(latas.shape[0]): # matriz
        for ii in range(latas.shape[1]): # linha
            paralelo[ii] = latas[i, ii, :]
        inversos = 1 / paralelo
        soma_inversos = np.sum(inversos)
        inverso_soma_inversos[i] = 1 / soma_inversos
    return inverso_soma_inversos

In [59]:
def aplanar_e_ordenar(latas):
    capacitores_com_posicao = []
    for i, ramo in enumerate(latas):
        for ii, linha in enumerate(ramo):
            for jj, cap in enumerate(linha):
                capacitores_com_posicao.append((cap, (i, ii, jj)))
    capacitores_com_posicao.sort(key=lambda x: x[0])
    return capacitores_com_posicao

def redistribuir(capacitores, n_mat, n_lin, n_col):
    latas_novas = [[[] for _ in range(n_lin)] for _ in range(n_mat)]
    posicoes = [[[] for _ in range(n_lin)] for _ in range(n_mat)]
    it = iter(capacitores)
    for i in range(n_mat):
        for ii in range(n_lin):
            for jj in range(n_col):
                cap, pos_original = next(it)
                latas_novas[i][ii].append(cap)
                posicoes[i][ii].append(pos_original)
    return latas_novas, posicoes

In [60]:
def criar_n_ser_otimizados(n_ser, posicoes_originais):
    n_ser_otimizados = [[[] for _ in range(n_lin)] for _ in range(n_mat)]
    for i in range(n_mat):
        for ii in range(n_lin):
            for jj in range(n_col):
                pos_original = posicoes_originais[i][ii][jj]
                num_serie_original = n_ser[pos_original[0]][pos_original[1]][pos_original[2]]
                n_ser_otimizados[i][ii].append(num_serie_original)
    return n_ser_otimizados

In [61]:
# Parâmetros da matriz de capacitores
n_col = 1
n_lin = 2
n_mat = 6

# Número total de capacitâncias
num_capacitancias = 12 * 2 * 3 * n_col * n_lin
df = pd.read_excel('meu_arquivo.xlsx')
cap_temp = df["uF"]
num_ser = df["no_serie"]

In [62]:
# Criar as matrizes de capacitores
capacitancias_por_matriz = n_col * n_lin
latas = np.zeros((n_mat, n_lin, n_col))
n_ser = np.zeros((n_mat, n_lin, n_col))

cont = 0
for i in range(n_mat):
    for ii in range(n_lin):
        for jj in range(n_col):
            n_ser[i, ii, jj] = num_ser[cont]
            latas[i, ii, jj] = cap_temp[cont]
            cont = cont + 1

In [63]:
x = capacitancia_equivalente_por_ramo(latas)
print(x)

[ 9.18918919  9.6        10.74418605  9.          8.97222222  7.96875   ]


/tmp/ipython-input-1639692738.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  paralelo[ii] = latas[i, ii, :]


In [64]:
# Primeiro aplanar e ordenar os capacitores
capacitores_ordenados = aplanar_e_ordenar(latas)

# Agora, redistribuir os capacitores ordenados
latas_reorganizadas, posicoes_originais = redistribuir(capacitores_ordenados, n_mat, n_lin, n_col)

latas_reorganizadas = np.array(latas_reorganizadas)

# # Calcular as novas capacitâncias equivalentes
capacitancias_equivalentes_velhas = capacitancia_equivalente_por_ramo(latas)
capacitancias_equivalentes_novas = capacitancia_equivalente_por_ramo(latas_reorganizadas)

print(capacitancias_equivalentes_velhas)
print(capacitancias_equivalentes_novas)

[ 9.18918919  9.6        10.74418605  9.          8.97222222  7.96875   ]
[ 7.74193548  8.5         8.74285714  9.24324324 10.24390244 11.47826087]


/tmp/ipython-input-1639692738.py:6: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  paralelo[ii] = latas[i, ii, :]
